In [1]:
import pandas as pd
from fuzzywuzzy import process, fuzz
import re

# Inlezen van de data
excel_file = '../../../data/H4/normalisering/machinaal_test.xlsx'
excel_data = pd.read_excel(excel_file)

csv_file = '../../../data/H4/normalisering/hist_antwerp_streets.csv'
csv_data = pd.read_csv(csv_file, sep=';')

# Functie om straatnamen op te schonen, inclusief spaties verwijderen
def clean_street_name(name):
    if pd.isnull(name):
        return ""
    name = str(name).lower()
    name = re.sub(r'^(de|het|van|den)\s+', '', name)
    name = re.sub(r'\bstr\b$', 'straat', name)
    name = re.sub(r'\d+', '', name)
    name = re.sub(r'[^\w\s;]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    name = name.replace(' ', '')  # <-- spaties verwijderen
    return name

# Voor individuele straatnamen in meervoudige vermeldingen
def clean_individual_street(name):
    if pd.isnull(name):
        return ""
    name = str(name).strip()
    name = re.sub(r'[^\w\s]', '', name)
    name = re.sub(r'\d+', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    name = name.replace(' ', '')  # <-- spaties verwijderen
    return name

# Schoonmaken van straatnamen
excel_data['clean_E_LOC'] = excel_data['E_LOC'].apply(clean_street_name)
csv_data['clean_straatnaam'] = csv_data['streetname'].apply(clean_street_name)

# Lijst met unieke opgeschoonde straatnamen uit referentie
csv_street_list = csv_data['clean_straatnaam'].dropna().unique().tolist()

# Functie om de beste match te zoeken
def get_best_match(street_name, choices, score_cutoff=70):
    if pd.isnull(street_name) or street_name == "":
        return None, 0
    best_match = process.extractOne(
        street_name, 
        choices,
        scorer=fuzz.ratio,
        score_cutoff=score_cutoff
    )
    if best_match:
        return best_match
    return None, 0

# Initialiseren van nieuwe kolommen
excel_data['E_LOC_gis1898'] = None
excel_data['E_LOC_lat'] = None
excel_data['E_LOC_long'] = None
excel_data['matched_street'] = None
excel_data['match_score'] = 0

# Itereren over de rijen en matches zoeken
for idx, row in excel_data.iterrows():
    clean_value = row['clean_E_LOC']
    street_parts = [clean_individual_street(part) for part in clean_value.split(';')]
    
    match_found = False
    for street_part in street_parts:
        if street_part == "":
            continue
            
        match_result, score = get_best_match(street_part, csv_street_list)
        
        if match_result:
            matched_idx = csv_data[csv_data['clean_straatnaam'] == match_result].index[0]
            matched_street_data = csv_data.loc[matched_idx]
            
            excel_data.at[idx, 'E_LOC_gis1898'] = matched_street_data['streetname']
            excel_data.at[idx, 'E_LOC_lat'] = matched_street_data['lat']
            excel_data.at[idx, 'E_LOC_long'] = matched_street_data['long']
            excel_data.at[idx, 'matched_street'] = matched_street_data['streetname']
            excel_data.at[idx, 'match_score'] = score
            
            match_found = True
            break

    if not match_found and pd.notna(row['E_LOC']) and row['E_LOC'].strip() != "":
        excel_data.at[idx, 'matched_street'] = "unknown"
        excel_data.at[idx, 'E_LOC_gis1898'] = "unknown"
        excel_data.at[idx, 'E_LOC_lat'] = None
        excel_data.at[idx, 'E_LOC_long'] = None
        excel_data.at[idx, 'match_score'] = 0

# Kolommen exporteren naar Excel
final_columns = [col for col in excel_data.columns if col not in ['clean_E_LOC', 'matched_street', 'match_score']]
excel_data[final_columns].to_excel('../../../data/H4/normalisering/machinaal_tes_loc.xlsx', index=False)

# Statistische samenvatting
total_records = len(excel_data)
valid_records = excel_data['E_LOC'].notna().sum()
matched_records = len(excel_data[excel_data['matched_street'].notna()])
high_confidence_matches = len(excel_data[excel_data['match_score'] >= 90])
medium_confidence_matches = len(excel_data[(excel_data['match_score'] >= 75) & (excel_data['match_score'] < 90)])
low_confidence_matches = len(excel_data[(excel_data['match_score'] >= 70) & (excel_data['match_score'] < 75)])
multiple_parts_count = sum(excel_data['E_LOC'].str.contains(';', na=False))

print(f"Total records processed: {total_records}")
print(f"Records with valid street names (non-NaN): {valid_records}")
print(f"Records with multiple street names (containing ';'): {multiple_parts_count}")
print(f"Records with matches found: {matched_records} ({matched_records/valid_records*100:.1f}%)")
print(f"High confidence matches (>=90): {high_confidence_matches}")
print(f"Medium confidence matches (75-89): {medium_confidence_matches}")
print(f"Low confidence matches (70-74): {low_confidence_matches}")
print(f"No matches found: {valid_records - matched_records}")


Total records processed: 14179
Records with valid street names (non-NaN): 9011
Records with multiple street names (containing ';'): 2534
Records with matches found: 9011 (100.0%)
High confidence matches (>=90): 5563
Medium confidence matches (75-89): 2558
Low confidence matches (70-74): 442
No matches found: 0
